# 00 · Data preparation

**Question.** Which 480 pipeline runs does the study contain, with what settings, and where is each run's VCF?

**Reviewer comment.** None.

**Produces.** `results/00_data_preparation/tables/run_manifest.csv`: one row per run, the `TestCases.csv` metadata joined to the run's VCF path by `TestCaseNo`. Notebook 01 reads it.

**Does not.**
- parse any VCF record (notebook 01);
- cover the indel runs (`TestCases_Indels.csv` has 13 columns and 320 runs);
- build the truth set or compute any metric.

**Legacy join.** The legacy notebook matched a `TestCases.csv` row to a VCF folder by row position (`int(key) - 1 == index`). Here the join is on `canon(TestCaseNo)`, and the notebook asserts that both agree for every row.

Run from `notebooks/`. `SWB_DATA_ROOT` points at the external drive.

## Imports

In [1]:
import sys
from pathlib import Path

try:
    import swb  # noqa: F401
except ImportError:  # swb is not pip-installed: use the repository's src/ (notebooks run from notebooks/)
    sys.path.insert(0, str(Path.cwd().parent / "src"))

import os

import pandas as pd

from swb import audit, config, io

## Config

In [2]:
OUT = config.results_dir("00_data_preparation")
STAGE = audit.Stage(OUT / ".staging")

EXPECTED_RUNS = 480
LEVELS = {
    "Sample": ["EA", "FD", "IL", "LL", "NC"],
    "Environment": ["Altay + COSAP", "Uhem + COSAP"],
    "isTrimmed": ["NO", "YES"],
    "baseRecalibration": ["NO", "YES"],
    "Mapper": ["BOWTIE", "BWA"],
    "VariantCaller": ["Mutect", "SomaticSniper", "Strelka"],
    "Duplicates": ["DELETE", "MARK"],
}
print("SWB_DATA_ROOT:", config.DATA_ROOT)

SWB_DATA_ROOT: /Volumes/E4 Pro/Bioinformatics-StabilityAnalysis


## Load + validate

In [3]:
for p in (config.DATA_ROOT, config.VCF_DIR, config.METADATA_CSV, config.TRUTH_VCF, config.EXOME_BED):
    assert p.exists(), "missing input {} (is the external drive mounted? set SWB_DATA_ROOT)".format(p)
assert config.TRUTH_VCF.stat().st_size > 0 and config.EXOME_BED.stat().st_size > 0

# VCFs: real files only ('._' AppleDouble files are skipped), one per TestCase folder
vcf_files = io.vcf_filenames(str(config.VCF_DIR))
n_appledouble = sum(n.startswith("._") for _, _, names in os.walk(str(config.VCF_DIR)) for n in names)
assert len(vcf_files) == EXPECTED_RUNS, len(vcf_files)
assert len(io.real_files(str(config.VCF_DIR))) == len(vcf_files), "a TestCase folder holds more than one VCF"

# Metadata: schema, keys, and the row-position assumption the legacy join relied on
raw = pd.read_csv(config.METADATA_CSV)
assert list(raw.columns) == config.METADATA_COLUMNS, list(raw.columns)
raw["key"] = raw["TestCaseNo"].map(io.canon)
assert raw["key"].is_unique
assert (raw["key"].astype(int) == raw.index + 1).all(), "TestCaseNo no longer equals row position + 1"
assert set(vcf_files) <= set(raw["key"]), "VCF folders without a metadata row"

# The 480 runs: factor levels, no missing values, one run per factor combination
runs = raw[raw["key"].isin(vcf_files)].rename(columns=config.METADATA_RENAME)
assert len(runs) == EXPECTED_RUNS, len(runs)
for col, levels in LEVELS.items():
    assert runs[col].notna().all(), "missing values in " + col
    assert sorted(runs[col].unique()) == levels, (col, sorted(runs[col].unique()))
assert runs["phase"].eq("done").all() and runs["ReferenceGenom"].eq("hg38").all()
assert runs.groupby(list(LEVELS)).ngroups == EXPECTED_RUNS, "not one run per factor combination"
assert runs[config.METADATA_COMPUTED_COLUMNS].isna().all().all()

print(len(raw), "TestCases.csv rows,", len(vcf_files), "VCFs,", n_appledouble, "AppleDouble files skipped")

864 TestCases.csv rows, 480 VCFs, 405 AppleDouble files skipped


## Analysis

In [4]:
manifest = runs.drop(columns=config.METADATA_COMPUTED_COLUMNS).copy()
manifest["TestCaseNo"] = manifest["key"]
manifest = manifest.drop(columns="key")
manifest.insert(0, "run", manifest["TestCaseNo"].astype(int))
manifest["vcf_relpath"] = manifest["TestCaseNo"].map(
    lambda k: os.path.relpath(vcf_files[k], str(config.DATA_ROOT)))
manifest = manifest.sort_values("run").reset_index(drop=True)
manifest.head()

,run,TestCaseNo,phase,Sample,operatingSystem,Environment,isTrimmed,baseRecalibration,Mapper,M_Version,...,VariantFilter,FilterPass,SOMATICfilter,remove_indels,FileLink,Author,Slurm_id,Node,ElapsedTime,vcf_relpath
0,2,2,done,EA,Linux,Altay + COSAP,YES,YES,BWA,NaN,...,NaN,NaN,NaN,NaN,https://drive.google.com/drive/folders/11CFGjN...,GUL EDA,29812.0,a047,07:15:00,vcf/TESTCASES_bedded/TestCase 2/snp_SRR7890919...
1,8,8,done,EA,Linux,Altay + COSAP,NO,NO,BWA,NaN,...,NaN,NaN,NaN,NaN,https://drive.google.com/drive/folders/1I7r5au...,GUL EDA,29812.0,a047,06:12:47,vcf/TESTCASES_bedded/TestCase 8/snp_SRR7890919...
2,14,14,done,EA,Linux,Altay + COSAP,YES,NO,BWA,NaN,...,NaN,NaN,NaN,NaN,https://drive.google.com/drive/folders/1AN-msy...,GUL EDA,29812.0,a047,05:48:51,vcf/TESTCASES_bedded/TestCase 14/snp_SRR789091...
3,20,20,done,EA,Linux,Altay + COSAP,NO,YES,BWA,NaN,...,NaN,NaN,NaN,NaN,https://drive.google.com/drive/folders/1hhxI84...,GUL EDA,29812.0,a047,07:44:08,vcf/TESTCASES_bedded/TestCase 20/snp_SRR789091...
4,26,26,done,EA,Linux,Altay + COSAP,YES,YES,BOWTIE,NaN,...,NaN,NaN,NaN,NaN,https://drive.google.com/drive/folders/1sxq7kO...,GUL EDA,29811.0,a078,07:36:41,vcf/TESTCASES_bedded/TestCase 26/snp_SRR789091...


## Save tables

In [5]:
io.write_csv(manifest, STAGE.path(OUT / "tables" / "run_manifest.csv"), sort_by="run")

## Figures

None.

## Parity check

Against the legacy caches `vcfcomparison_df_full.csv` and `vcf_filenames.csv`. Results are moved out of staging only if every check passes.

In [6]:
legacy_meta = pd.read_csv(config.LEGACY_METADATA)
legacy_meta["TestCaseNo"] = legacy_meta["TestCaseNo"].map(io.canon)
legacy_meta = legacy_meta.set_index("TestCaseNo")
legacy_files = pd.read_csv(config.LEGACY_FILENAMES)
legacy_paths = dict(zip(legacy_files["testcase"].astype(str), legacy_files["filepath"]))

ours = manifest.set_index("TestCaseNo")
parity = audit.Parity()
parity.check("run keys == legacy vcfcomparison_df_full", set(ours.index) == set(legacy_meta.index),
             "{} runs".format(len(ours)))
parity.check("VCF paths == legacy vcf_filenames.csv", ours["vcf_relpath"].to_dict() == legacy_paths)
shared = [c for c in ours.columns if c in legacy_meta.columns]
differs = {c: int((ours[c].astype(str) != legacy_meta[c].reindex(ours.index).astype(str)).sum()) for c in shared}
differs = {c: n for c, n in differs.items() if n}
parity.check("metadata columns == legacy ({} columns)".format(len(shared)), not differs, differs)

parity.finish(OUT / "audit" / "parity.txt", STAGE)

parity: PASS (3 checks)
PASS  run keys == legacy vcfcomparison_df_full  [480 runs]
PASS  VCF paths == legacy vcf_filenames.csv
PASS  metadata columns == legacy (21 columns)  [{}]



## Audit summary

In [7]:
lines = [
    "runs: {}".format(len(manifest)),
    "design: " + " x ".join("{} {}".format(len(v), k) for k, v in LEVELS.items()) + ", one run per combination",
    "TestCases.csv rows: {} (rows without a VCF folder are not runs)".format(len(raw)),
    "VCF folders with a real VCF: {}; AppleDouble '._' files skipped: {}".format(len(vcf_files), n_appledouble),
    "join: canon(TestCaseNo); equals the legacy row-position join for all {} rows".format(len(raw)),
    "truth VCF and exome BED present and non-empty",
    "python {} | pandas {}".format(sys.version.split()[0], pd.__version__),
]
(OUT / "audit").mkdir(parents=True, exist_ok=True)
(OUT / "audit" / "summary.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

runs: 480
design: 5 Sample x 2 Environment x 2 isTrimmed x 2 baseRecalibration x 2 Mapper x 3 VariantCaller x 2 Duplicates, one run per combination
TestCases.csv rows: 864 (rows without a VCF folder are not runs)
VCF folders with a real VCF: 480; AppleDouble '._' files skipped: 405
join: canon(TestCaseNo); equals the legacy row-position join for all 864 rows
truth VCF and exome BED present and non-empty
python 3.8.12 | pandas 2.0.0
